This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [17]:
import great_expectations as gx
context = gx.get_context()
import logging

INFO:great_expectations.data_context.data_context.file_data_context:FileDataContext loading fluent config
INFO:great_expectations.datasource.fluent.config:Loading 'datasources' ->
[{'assets': [...],
  'connection_string': 'bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?credentials_path=/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json',
  'create_temp_table': True,
  'name': 'gfw-google-827',
  'type': 'sql'}]
INFO:great_expectations.data_context.data_context.abstract_data_context:Usage statistics is disabled; skipping initialization.
INFO:great_expectations.data_context.data_context.abstract_data_context:Loaded 'gfw-google-827' from fluent config
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


In [18]:
logging.basicConfig(level=logging.INFO, force = True)

In [19]:
## THIS IS REQUIRED FOR THE TECHNICAL VIEW HACK
# TODO handling of credentials not ideal, required for technical view fix
import os

from google.cloud import bigquery
import pandas as pd

sa_credentials_path=os.environ['SA_CREDENTIALS_PATH']
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = sa_credentials_path
client = bigquery.Client()

In [20]:
gx_temp_schema = "tech_great_expectations_temp_ttl_7d"
date_filter = "hour <= '2099-12-31'"

In [21]:
def is_sharded_table(client, schema_name: str, table_name: str):
    # TODO CHO20230706 Implement more robust logic to determine whether table is sharded
    return(table_name.endswith("_"))


In [22]:
def partition_enforcement_enabled(client, schema_name: str, table_name: str):
    partition_enforcement_enabled_sql = f"""
  SELECT
    option_value
  FROM
    {schema_name}.INFORMATION_SCHEMA.TABLE_OPTIONS
  WHERE
    table_name = '{table_name}'
  AND 
    option_name = 'require_partition_filter';
    """
    
    # TODO CHO20230622 handle exceptions
    partition_enforcement_enabled_res = pd.read_gbq(
        partition_enforcement_enabled_sql, 
        project_id='world-fishing-827', 
        dialect='standard'
    )

    # TODO CHO20230622 handle multiple rows returned
    return(partition_enforcement_enabled_res.shape[0] > 0)

In [23]:
def get_partition_cols(client, schema_name: str, table_name: str):
    get_partition_cols_sql = f"""
  SELECT
    column_name,
    data_type,
    is_hidden
  FROM
    {schema_name}.INFORMATION_SCHEMA.COLUMNS
  WHERE
    table_name = '{table_name}'
  AND
    is_partitioning_column = 'YES';
    """
    
    # TODO CHO20230622 handle exceptions
    get_partition_cols_res = pd.read_gbq(
        get_partition_cols_sql, 
        project_id='world-fishing-827', 
        dialect='standard'
    )

    # TODO CHO20230622 handle multiple rows returned
    return(get_partition_cols_res)

In [24]:
def create_or_replace_tech_view(
    client, 
    dataset_name: str, 
    table_name: str, 
    gx_temp_schema: str, 
    date_filter_sql: str = None,
    sharded_table_select_shards_sql: str = None,
    select_splitter_col_sql: str = None
):

    # TODO CHO20230622 validate parameters, e.g. date filter
    # TODO CHO20230622 check first whether view exists before replacing
    fq_view_name = f"{gx_temp_schema}.v_unfiltered_{dataset_name}_{table_name}"
    query = f"""
    CREATE OR REPLACE VIEW `{fq_view_name}` AS (
    SELECT *{select_splitter_col_sql}
    FROM `{dataset_name}.{table_name}{sharded_table_select_shards_sql}` {date_filter_sql}
    )
    """
    logging.info(query)
    # TODO CHO20230705 handle exceptions better
    resp=client.query(query)
    logging.info(resp.result())
    return(fq_view_name)    

In [25]:
connection_string = f"""bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?\
credentials_path={sa_credentials_path}"""

In [26]:
import yaml

In [27]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [28]:
datasource_config.get("project")

'gfw-google-827'

In [29]:
gx_project = datasource_config.get("project")
#we create a data source for each schema, e.g. pipe_ais_v3_alpha_published
#get datasource if it exists, otherwise create datasource
# WARNING: it's necessary to distinguish because running add_or_update_sql resets the datasource config
# TODO: create feature request to simply get datasource if it already exists
if gx_project in [ds.get("name") for ds in context.list_datasources()]:
    gx_datasource = context.get_datasource(gx_project)
else:
    gx_datasource = context.sources.add_or_update_sql(
        name=gx_project, connection_string=connection_string, create_temp_table=True
    )

In [30]:
def add_datasource(
        client, 
        datasource_name, 
        dataset_name, 
        table_name, 
        version_number, 
        gx_temp_schema, 
        gx_datasource,
        default_max_date: str = "2099-12-31",
        asset_exists_behaviour: str = ['update', 'skip'][0]
    ):

    asset_name=f"{datasource_name}-{version_number}"
    if asset_name in gx_datasource.get_asset_names():
        if asset_exists_behaviour == 'skip':
            logging.info(f"{asset_name} already exists. Skipping!")
            return
        else:
            logging.info(f"{asset_name} already exists. Updating by removing it before adding again!")
            gx_datasource.delete_asset(asset_name)

    # TODO CHO20230705 handle non-date partition cols
    date_partition_cols = get_partition_cols(client, dataset_name, table_name).query("data_type.isin(['TIMESTAMP', 'DATE'])")
    partition_filter_enforced = partition_enforcement_enabled(client, dataset_name, table_name)
    table_is_sharded = is_sharded_table(client, dataset_name, table_name)

    date_filter_sql=''
    table_is_partitioned=None
    sharded_table_select_shards_sql=''
    select_splitter_col_sql=''
    has_splitter_col=False

    if table_is_sharded:
        select_splitter_col_sql=f", PARSE_DATE('%Y%m%d', _TABLE_SUFFIX) AS SPLITTER_COLUMN"
        has_splitter_col=True
        logging.info(f"""
            Sharded column _TABLE_SUFFIX is hidden and is therefore added to `SELECT *` statement
        """)
        sharded_table_select_shards_sql='*'
        
    # TODO CHO20230707: not necessary to apply dummy date filter in view, but might still 
    # make sense to apply splitter
    if not partition_filter_enforced:
        logging.info(f"Partition enforcement not enabled, not applying date filter")
    else:
        if not date_partition_cols.shape[0]:
            logging.warn(f"""
                Partition enforcement enabled but no date columns found.
                Proceeding but eventually queries will fail!
            """)
        else:
            if date_partition_cols.query("data_type=='TIMESTAMP'").shape[0]:
                splitter_col=date_partition_cols.query("data_type=='TIMESTAMP'")["column_name"][0]
                select_splitter_col_sql = f', DATE({splitter_col}) AS SPLITTER_COLUMN'
            else:
                splitter_col=date_partition_cols["column_name"][0]
                select_splitter_col_sql = f', {splitter_col} AS SPLITTER_COLUMN'

            date_filter_sql = f"WHERE {splitter_col} < '{default_max_date}'"
            has_splitter_col=True
            logging.info(f"""
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: {date_filter}
            """)

    gx_view_name = create_or_replace_tech_view(
        client=client, 
        dataset_name=dataset_name, 
        table_name=table_name, 
        gx_temp_schema=gx_temp_schema, 
        date_filter_sql=date_filter_sql,
        sharded_table_select_shards_sql=sharded_table_select_shards_sql,
        select_splitter_col_sql=select_splitter_col_sql
    )

    batch_metadata={
        'datasource_name': datasource_name, 
        'dataset_name': dataset_name, 
        'table_name': table_name, 
        'version_number': version_number
    }
        
    table_asset = gx_datasource.add_query_asset(
        name=asset_name,
        query=f"SELECT * FROM {gx_view_name}",
        batch_metadata=batch_metadata
    )

    # add splitter and sorter if table is sharded or partition column exists    
    if has_splitter_col:
        table_asset.add_splitter_column_value("SPLITTER_COLUMN")
        table_asset.add_sorters([f"-SPLITTER_COLUMN"])

In [36]:
for current_datasource in datasource_config.get("datasources"):
    datasource_name = current_datasource.get('name')
    logging.info(f"Adding datasource '{current_datasource.get('name')}'")
    if 'asset_exists_behaviour' in current_datasource:
            asset_exists_behaviour=current_datasource.get('asset_exists_behaviour')
    else:
        asset_exists_behaviour='skip'
    for current_version_number in current_datasource.get('versions'):
        logging.info(f"Adding version '{current_version_number}'")
        config_current_version = current_datasource.get('versions').get(current_version_number)
        dataset_name = config_current_version.get('dataset')
        table_name = config_current_version.get('table')
        add_datasource(
            client,
            datasource_name,
            dataset_name, 
            table_name, 
            current_version_number, 
            gx_temp_schema, 
            gx_datasource, 
            asset_exists_behaviour=asset_exists_behaviour
        )


INFO:root:Adding datasource 'messages'
INFO:root:Adding version '3.0.0'
INFO:root:messages-3.0.0 already exists. Skipping!
INFO:root:Adding version '2.5'
INFO:root:messages-2.5 already exists. Skipping!
INFO:root:Adding datasource 'satellite_timing_offsets'
INFO:root:Adding version '3.0.0'
INFO:root:satellite_timing_offsets-3.0.0 already exists. Skipping!
INFO:root:Adding version '2.5'
INFO:root:satellite_timing_offsets-2.5 already exists. Skipping!
INFO:root:Adding datasource 'segs_activity'
INFO:root:Adding version '3.0.0'
INFO:root:segs_activity-3.0.0 already exists. Skipping!
INFO:root:Adding version '2.5'
INFO:root:segs_activity-2.5 already exists. Skipping!
INFO:root:Adding datasource 'segs_activity_daily'
INFO:root:Adding version '3.0.0'
INFO:root:segs_activity_daily-3.0.0 already exists. Skipping!
INFO:root:Adding version '2.5'
INFO:root:segs_activity_daily-2.5 already exists. Skipping!
INFO:root:Adding datasource 'ssvids_identities'
INFO:root:Adding version '3.0.0'
INFO:root:s

In [32]:
gx_datasource.get_asset_names()

{'encounters-2.5',
 'encounters-3.0.0',
 'fragments-3.0.0',
 'messages-2.5',
 'messages-3.0.0',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_daily-3.0.0',
 'stats_daily-2.5',
 'stats_daily-3.0.0',
 'vessel_info-2.5',
 'vessel_info-3.0.0'}